# 07 -- Tree models (decision tree)

A single CART tree: non-linear and highly interpretable, but prone to
overfitting. This notebook tunes complexity controls and shows the
classic depth-vs-loss overfitting curve.

1. `GridSearchCV` over depth, leaf size, split criterion, class weight.
2. Depth **loss curve** (train vs validation log loss) and CV validation curve.
3. Holdout evaluation and feature importance.

In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.modeling import (
    classification_metrics,
    cv_results_frame,
    evaluate_split,
    load_modeling_data,
    make_pipeline,
    metrics_frame,
    run_grid_search,
    subsample,
    train_valid_report,
)
from src.modeling import plots

data = load_modeling_data()
pd.Series(data.summary())


In [ ]:
# Fraction of the TRAIN partition used for GridSearchCV. The holdout is never
# touched during tuning. Lower this (e.g. 0.3) for faster iteration, then set
# back to 1.0 to confirm the final model on the full training data.
SEARCH_FRAC = 1.0
search = subsample(data, frac=SEARCH_FRAC)
pd.Series(search.summary())


## Hyperparameter search

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=42)
param_grid = {
    "max_depth": [3, 4, 5, 6, 8, 10],
    "min_samples_leaf": [20, 50, 100, 200],
    "criterion": ["gini", "entropy"],
    "class_weight": [None, "balanced"],
}
gs = run_grid_search(tree, param_grid, search, refit="roc_auc")
cv_results_frame(gs).head(10)

## Loss curve across tree depth

A single tree has no iterations, so we sweep `max_depth` and record
train/validation log loss. The validation minimum marks the depth where
the tree starts to overfit.

In [ ]:
from src.modeling.curves import param_loss_curve
from sklearn.tree import DecisionTreeClassifier

depths = list(range(1, 17))
tr, va = param_loss_curve(
    lambda d: DecisionTreeClassifier(
        max_depth=d, min_samples_leaf=50, class_weight="balanced", random_state=42
    ),
    depths, search.X_train, search.y_train, data.X_holdout, data.y_holdout,
)
plots.plot_param_loss_curve(depths, tr, va, xlabel="max_depth", title="Decision tree depth loss curve")
plt.show()

## CV validation and learning curves

In [ ]:
from sklearn.tree import DecisionTreeClassifier

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plots.plot_validation_curve(
    make_pipeline(DecisionTreeClassifier(min_samples_leaf=50, class_weight="balanced", random_state=42)),
    search.X_train, search.y_train,
    param_name="model__max_depth",
    param_range=[2, 3, 4, 5, 6, 8, 10, 12],
    cv=search.cv, ax=axes[0],
)
plots.plot_learning_curve(gs.best_estimator_, search.X_train, search.y_train, cv=search.cv, ax=axes[1])
plt.tight_layout()
plt.show()

## Holdout evaluation

In [ ]:
# Train vs holdout metrics for the selected model (overfitting check)
report = train_valid_report(gs.best_estimator_, data, valid_as_holdout=True)
report.round(4)


In [ ]:
# Holdout diagnostics for the selected model
best = gs.best_estimator_
proba_ho = best.predict_proba(data.X_holdout)[:, 1]

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
plots.plot_roc_pr(data.y_holdout.to_numpy(), proba_ho, label="tree", axes=axes[0])
plots.plot_calibration(data.y_holdout.to_numpy(), proba_ho, ax=axes[1][0], label="tree")
plots.plot_score_distribution(data.y_holdout.to_numpy(), proba_ho, ax=axes[1][1])
plt.tight_layout()
plt.show()

best_t = classification_metrics(data.y_holdout.to_numpy(), proba_ho)["best_f1_threshold"]
fig, ax = plt.subplots(figsize=(5, 5))
plots.plot_confusion(data.y_holdout.to_numpy(), proba_ho, threshold=best_t, ax=ax)
plt.show()


## Feature importance

In [ ]:
# Feature importance for the selected model
pre = gs.best_estimator_.named_steps["preprocess"]
feat_names = list(pre.transform(search.X_train.iloc[:1]).columns)
model = gs.best_estimator_.named_steps["model"]
importances = getattr(model, "feature_importances_", None)
if importances is not None:
    fig, ax = plt.subplots(figsize=(8, 8))
    plots.plot_feature_importance(feat_names, importances, top=20, ax=ax, title="Decision tree importance")
    plt.show()


## Takeaways

- The depth loss curve shows the overfitting point; CV picks a shallow,
  regularised tree.
- A single tree typically trails logistic regression on AUC but captures
  interactions; ensembles and boosting build on this.